# SEVIRI SSL: dataset sampling and coverage

This notebook documents the observations underlying the [SSL training comparison](study_seviri_ssl_training.ipynb): clip contents, geographic crop origins, observation dates and normalization. It reads the prepared manifests and source-grid coordinates. The historical sequence has five nested training sets up to 65,536 clips; the 131,072-clip dataset prepared on 2 November uses resampled origins and a stricter geographic footprint. A second 131,072-clip dataset, prepared on 10 November, samples January–June 2022 with equal monthly contributions and has its own validation manifests. Small and Base share the prepared dataset at a given size. The [CMA/CT/RDT study](study_seviri_cloud_mask_probe.ipynb) documents its separate July 2022 probing dataset.

Execute cells in order with **Python (deep_convection)**. Tables are optional through `SHOW_TABLES`; figures and numerical exports are written to `results/dataset_study_20261110/`.

In [ ]:
from pathlib import Path
import importlib, json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/emma_gpm").is_dir())
sys.path.insert(0, str(ROOT / "src"))
SERIES = ROOT / "results/ssl_scaling_20261025"
SERIES_PATHS = [SERIES, ROOT / "results/ssl_extension_20261026", ROOT / "results/ssl_base_32768_20261026"]
LEARNING_SERIES_PATHS = SERIES_PATHS + [ROOT / "results/ssl_factorial_20261028"]
OUTPUT = ROOT / "results/dataset_study_20261110"
OUTPUT.mkdir(exist_ok=True)
SHOW_TABLES = False
if "monitor" in globals():
    monitor.close()
from emma_gpm import ssl_study, ssl_dashboard
study = importlib.reload(ssl_study)
views = importlib.reload(ssl_dashboard)
loaded_studies = [study.load_study(path) for path in SERIES_PATHS if (path / "scaling_plan.json").exists()]
SNAPSHOT = study.merge_studies(*loaded_studies)
learning_studies = loaded_studies + [study.load_study(path) for path in LEARNING_SERIES_PATHS
                                   if path not in SERIES_PATHS and (path / "scaling_plan.json").exists()]
LEARNING_SNAPSHOT = study.merge_studies(*learning_studies)
GRID_COMPLETION = ROOT / "results/ssl_grid_completion_20261103"
if (GRID_COMPLETION / "scaling_plan.json").exists():
    LEARNING_SNAPSHOT = study.add_completed_experiments(LEARNING_SNAPSHOT, study.load_study(GRID_COMPLETION))
# The main notebook presents SSL training. Historical probes have a separate notebook.
LEARNING_SNAPSHOT = {**LEARNING_SNAPSHOT, "runs": {
    key: run for key, run in LEARNING_SNAPSHOT["runs"].items()
    if run["stage"]["kind"] == "pretraining"}}
SNAPSHOT = LEARNING_SNAPSHOT
GEOGRAPHIC_SERIES = next((ROOT / name for name in (
    "results/ssl_131072_verified_20261102", "results/ssl_131072_20261102")
    if (ROOT / name / "scaling_plan.json").exists()), ROOT / "results/ssl_131072_verified_20261102")
GEOGRAPHIC_SNAPSHOT = (study.load_study(GEOGRAPHIC_SERIES)
                       if (GEOGRAPHIC_SERIES / "scaling_plan.json").exists() else None)

LEONARDO_TRAINING = Path("/data1/deep_convection_results/leonardo_base_future_131072_20261108/training")
if GEOGRAPHIC_SNAPSHOT and (LEONARDO_TRAINING / "progress.json").exists():
    GEOGRAPHIC_SNAPSHOT = study.attach_distributed_training(GEOGRAPHIC_SNAPSHOT, LEONARDO_TRAINING)

VALIDATION_STUDIES = {"Historical validation · 512 clips": LEARNING_SNAPSHOT}
if GEOGRAPHIC_SNAPSHOT:
    VALIDATION_STUDIES["Resampled validation · 512 clips"] = GEOGRAPHIC_SNAPSHOT

def table(frame):
    if not SHOW_TABLES:
        return
    if frame.empty:
        print("This comparison has no saved results in the selected snapshot.")
        return
    formats = {column: "{:.0f}" if pd.api.types.is_integer_dtype(frame[column]) else "{:.2f}"
               for column in frame.select_dtypes(include="number").columns}
    for column in frame.columns:
        if any(term in column.lower() for term in ("epoch", "clips", "updates", "models", "budget")):
            if pd.api.types.is_numeric_dtype(frame[column]):
                formats[column] = "{:.0f}"
        if "(%)" in column or column in ("Mean BT (K)", "BT standard deviation (K)"):
            formats[column] = "{:.1f}"
    display(frame.style.format(formats, na_rep="—").hide(axis="index"))

def figure(fig, filename):
    if fig is not None:
        fig.savefig(OUTPUT / filename, dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

display(Markdown(f"**Snapshot:** {LEARNING_SNAPSHOT['captured_at'][:19].replace('T', ' ')} Europe/Paris · "
                 f"**Learning-curve series:** {len(learning_studies)} · **Figures and CSV files:** `{OUTPUT.relative_to(ROOT)}`"))

## Observations, sampling units and forecast times

A **clip** is a sampling unit identified by its first timestamp and the origin of a spatial crop. Each clip contains 16 observations separated by five minutes, seven thermal channels and a 224 × 224 pixel crop. Its array shape is `(time, channel, y, x) = (16, 7, 224, 224)`. The elapsed time between its first and last observation is 75 minutes. Every selected clip contains finite observations at all required timestamps, channels and pixels, within the accepted brightness-temperature interval of 100–400 K. The reader uses the coordinate axes to orient each crop with north at the top and west on the left.

The channel order is **WV_062, WV_073, IR_087, IR_097, IR_108, IR_120, IR_134**. These identifiers denote the water-vapour and infrared channels at nominal wavelengths 6.2, 7.3, 8.7, 9.7, 10.8, 12.0 and 13.4 μm. In this notebook, **IR108** is the shorter name for IR_108.

The first ten frames form the historical input, spanning 45 minutes. The timestamp of the tenth frame is the **forecast reference time**. Each forecasting model receives this history and predicts two images, at **+15 and +30 minutes** relative to the reference time. In the stored clip, the historical frames have indices 0–9 and the targets have indices 12 and 15. The intermediate future observations remain available in the archive.

Masked reconstruction uses the same ten historical frames. This common input interval makes the information available to the four encoders comparable. All 16 stored frames contribute to clip validation and to the training-set normalization statistics.

## Spatial sampling and temporal separation

The historical 4,096-clip manifest contains observations from **12 and 13 September 2020**. The 8,192-clip manifest retains those clips and adds observations from **15 September**. The 16,384- and 32,768-clip manifests successively add combinations of time window and crop origin within those three dates. The 65,536-clip manifest also includes 12 October 2010, 13 April 2012, 30 September 2015 and 28 February 2016. These historical datasets form a nested sequence and share 512 validation clips from 17 September 2020.

The historical candidate origins use a **112-pixel spatial grid**, with an additional origin at the last admissible position along each image edge. This concentrates repeated geographic structures at a small number of positions relative to the crop. Each clip is identified by its first timestamp and spatial origin. Overlapping clips share observations and form correlated sampling units.

The **2 November 131,072-clip protocol** resamples all training origins at **one-pixel resolution**. It adds six observation dates: 5 November 2011, 18 November 2013, 7 and 30 November 2014, 16 November 2017 and 10 November 2019. Proposals are balanced across UTC dates and six-hour bins. Within each group, every physically supported combination of start time and pixel origin has equal proposal probability. Final training selection retains a 50% uniform component and a 50% component stratified by historical IR108 variability, with relative weights 1:2:4 for low, medium and high variability. The final distribution therefore also reflects radiometric variability. Spatial and temporal overlap within training is allowed. Each clip retains sixteen consecutive observations at five-minute cadence, and training order is shuffled at every epoch.

The agreed geographic domain is **20°W–40°E, 30–70°N**. Acquisition saves a rectangle in the satellite projection containing this geographic box; the saved rectangle extends beyond its geographic boundaries. An audit of the historical 65,536-clip manifest finds **33,235 crops entirely inside the agreed box**, along with **265 of its 512 validation crops**. The 2 November protocol checks every pixel centre in the complete 224×224 footprint against the geographic bounds. This condition applies to both training and validation. Its validation contains 512 randomly positioned crops from 17 September 2020, with at least 24 hours of temporal separation from the last training observation. Small and Base share this exact manifest. The 2 November protocol changes sampling, geographic coverage and validation along with training-set size; its learning curves retain a distinct validation-population label in the training notebook.

Both architectures use lossless native-frame files for the 2 November dataset, extracting crops with their original Kelvin values, spectral order and north-up, west-left orientation. The selected storage format follows a runtime comparison using the same clips, masks and model. The comparison and its initial cache-warming cost are recorded with the execution artifacts. A single global mean and standard deviation are estimated from all pixels of the selected training clips. The earlier series preserve their original normalization and storage metadata.

The spatial panels locate the **(0,0) pixel of each crop after orientation to north-up and west-left**. Each dot marks a distinct geographic origin, with colour indicating the percentage of clips in that split using that origin, aggregated over observation times. The dashed outline marks the agreed geographic domain. Each point is transformed using its source day's coordinates and satellite projection, preserving the alignment of the 763-row and 764-row grids. The displayed origins leave room for the entire crop within the applicable sampling domain. Historical maps retain all origins actually selected under the earlier protocol; the new map displays origins accepted by the full-footprint geographic check. The following maps show the historical 65,536-clip dataset and the 2 November resampled 131,072-clip dataset, each with its own validation origins.


### Historical variability and global normalization

The **historical variability score** is the mean absolute IR108 temperature change between successive five-minute observations, averaged over the nine historical frame pairs and all crop pixels. It is expressed in kelvin per observed five-minute difference and responds to transport and radiometric evolution at fixed pixels.

Each dataset size has one global mean and one global standard deviation calculated over every training clip, all 16 frames, seven channels and all crop pixels. Shared observations contribute once for each occurrence in a selected clip. These two constants transform both training and validation observations for that dataset size. The stored precision is used for computation; displayed temperatures and physical errors use rounded values. Reported metrics are converted back to Kelvin units.

In [ ]:
dataset_records = []
dataset_frames = {}
seen = set()
for population, snapshot in VALIDATION_STUDIES.items():
    for name, case in snapshot["cases"].items():
        key = (population, case["train_samples"])
        if key in seen:
            continue
        seen.add(key)
        train = pd.read_csv(case["queue"] / "dataset/train.csv")
        validation = pd.read_csv(case["queue"] / "dataset/validation.csv")
        times = pd.to_datetime(train.start_time, utc=True)
        validation_times = pd.to_datetime(validation.start_time, utc=True)
        gap = validation_times.min() - (times.max() + pd.Timedelta(minutes=75))
        dataset_frames[case["train_samples"]] = train
        dataset_records.append({"Training clips": len(train), "Validation clips": len(validation),
            "Training dates": times.dt.normalize().nunique(), "Distinct start times": times.nunique(),
            "Temporal separation (hours)": gap.total_seconds() / 3600,
            "Mean BT (K)": case["stats"]["global_mean_k"],
            "BT standard deviation (K)": case["stats"]["global_std_k"],
            "Validation population": population, "validation_sha256": snapshot["validation_sha256"]})
dataset_metadata = pd.DataFrame(dataset_records).sort_values("Training clips")
dataset_metadata.to_csv(OUTPUT / "dataset_metadata.csv", index=False)
table(dataset_metadata)
for row in dataset_metadata.to_dict("records"):
    display(Markdown(f"**{row['Training clips']:,} training clips:** {row['Training dates']} observation dates, "
        f"{row['Distinct start times']:,} distinct start times; global BT mean {row['Mean BT (K)']:.1f} K "
        f"and standard deviation {row['BT standard deviation (K)']:.1f} K."))

for snapshot, case_name, label in (
    (LEARNING_SNAPSHOT, "small_n65536", "historical_65536"),
    (GEOGRAPHIC_SNAPSHOT, "small_n131072", "resampled_131072")):
    if snapshot and case_name in snapshot["cases"]:
        coverage_figure, origins = views.dataset_overview(snapshot["cases"][case_name]["queue"], show_tables=SHOW_TABLES)
        coverage_figure.savefig(OUTPUT / f"{label}_crop_origins.png", dpi=120, bbox_inches="tight")
        for split, points in origins.items():
            points.to_csv(OUTPUT / f"{label}_origins_{split}.csv", index=False)


## Observation periods and repeated sampling

The heat map shows the percentage of each training set drawn from each UTC observation date. Every column sums to 100%; a zero entry means that date contributes no clips to that set. The following curves count observation dates and distinct first timestamps across all crops. The horizontal labels use **K = 1,024 clips**.

Multiple spatial crops can share the same timestamp, and neighbouring windows can share frames. Clip count therefore measures selected space–time windows, while the number of dates and first timestamps describes their temporal coverage. The 65,536-clip set spans seven dates and the 2 November 131,072-clip set thirteen dates. The calendar-balanced dataset described below has a separate distribution. Assessing diversity of atmospheric situations also requires examining the underlying weather episodes and spatial overlap.

The two historical 512-clip validation populations both use 17 September 2020 and have different crop selections. Leonardo's additional 10,240-clip validation is identified separately in the [training notebook](study_seviri_ssl_training.ipynb); its scores retain their own evaluation population.

In [ ]:
import numpy as np
counts = pd.DataFrame({size: pd.to_datetime(frame.start_time, utc=True).dt.strftime("%Y-%m-%d").value_counts()
                       for size, frame in sorted(dataset_frames.items())}).fillna(0).sort_index()
fractions = counts.div(counts.sum(axis=0), axis=1) * 100
fig, axis = plt.subplots(figsize=(10, 6), layout="constrained")
image = axis.imshow(fractions.to_numpy(), aspect="auto", cmap="viridis", vmin=0)
axis.set(xticks=range(len(counts.columns)), xticklabels=[f"{n:,}" for n in counts.columns],
         yticks=range(len(counts.index)), yticklabels=counts.index,
         xlabel="Training clips", ylabel="Observation date (UTC)", title="Training observations by date")
fig.colorbar(image, ax=axis, label="Training clips from this date (%)")
figure(fig, "training_date_distribution.png")
counts.to_csv(OUTPUT / "training_clips_by_date.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
for axis, metric in zip(axes, ("Training dates", "Distinct start times")):
    axis.plot(dataset_metadata["Training clips"], dataset_metadata[metric], "o-", color="#0072B2")
    axis.set(title=metric, xlabel="Training clips", ylabel=metric)
    axis.set_xscale("log", base=2)
    axis.set_xticks(dataset_metadata["Training clips"], [f"{n // 1024}K" for n in dataset_metadata["Training clips"]])
    axis.grid(alpha=.2)
figure(fig, "temporal_sampling_units.png")


## Original extended validation: 10,240 clips

The extended validation prepared on 2 November contains 512 clips from each of 20 UTC dates between 5 January and 15 June 2022. Each date contributes 128 clips in each six-hour start-time interval. These dates are separated from the pretraining observations by at least 24 hours. Spatial origins were drawn at one-pixel resolution, with the complete 224 × 224 crop inside 20°W–40°E and 30–70°N.

The map locates pixel (0, 0) of each north-up, west-left crop. Its colour gives the percentage of the 10,240 clips sharing that geographic origin. The date and hour panels count clips by their first timestamp; the hourly counts reflect both random selection and the availability of complete 75-minute windows. The training notebook evaluates this fixed validation population using the normalization of each evaluated encoder.


In [ ]:
EXTENDED = ROOT / "results/ssl_validation_10k_20261102"
extended_protocol = json.loads((EXTENDED / "protocol.json").read_text())
extended_manifest = pd.read_csv(EXTENDED / "validation.csv")
coverage, extended_origins = views.manifest_coverage_figure(
    extended_manifest, extended_protocol["zarr_root"], title="Extended validation · 10,240 clips")
figure(coverage, "extended_validation_10240_coverage.png")
extended_origins.to_csv(OUTPUT / "extended_validation_10240_origins.csv", index=False)
pd.to_datetime(extended_manifest.start_time, utc=True).dt.normalize().value_counts().sort_index().rename("clips").to_csv(
    OUTPUT / "extended_validation_10240_dates.csv")


## Calendar-balanced dataset: January–June 2022

The dataset prepared on 10 November contains **131,072 training clips from 93 UTC dates**, between 1 January and 29 June 2022. Each of the six months contributes approximately one sixth of the clips. Within a month, clips are allocated equally across eligible dates and then across four six-hour start-time intervals. Integer quotas differ by at most one clip at each allocation step. The monthly chart also displays July–December to show the coverage over the calendar year.

Training uses **three randomly selected start times per six-hour interval on each date**, giving 1,116 distinct start times. For each selected time, crop origins are sampled uniformly at one-pixel resolution among footprints satisfying the geographic and data-quality conditions. Multiple crops share each time window. Every retained 224 × 224 footprint lies within 20°W–40°E and 30–70°N, and contains complete observations at all 16 timestamps and seven channels. The spatial map shows each crop's north-up, west-left pixel (0, 0).

The **calendar-balanced validation** contains 10,240 clips on the same 20 reserved dates shown for the original extended validation. It uses newly sampled crops and start times, with equal monthly contributions and equal allocations across dates within each month and six-hour intervals within each date. Consequently, each date's contribution depends on the number of reserved dates in its month. A fixed, month-balanced subset of 512 clips supplies validation after every epoch and checkpoint selection; all 10,240 clips are evaluated every ten epochs. The original extended-validation manifest remains a separately identified reference population.

Training and validation windows are separated by at least 24 hours. Dates reserved for the independent probing datasets and their temporal separation intervals are also excluded from training. A single mean and standard deviation, calculated across all pixels, frames and channels of the 131,072 training clips, normalize both splits. The printed statistics below use kelvin units.

In [ ]:
CALENDAR = Path("/data1/deep_convection_data/ssl_calendar_131072_20261110")
calendar_protocol = json.loads((CALENDAR / "training/dataset/protocol.json").read_text())
calendar_stats = json.loads((CALENDAR / "training/dataset/channel_stats.json").read_text())
calendar_manifests = {
    "Training · 131,072 clips": pd.read_csv(CALENDAR / "training/dataset/train.csv"),
    "Extended validation · 10,240 clips": pd.read_csv(CALENDAR / "extended/dataset/validation.csv"),
    "Primary validation · 512 clips": pd.read_csv(CALENDAR / "training/dataset/validation.csv"),
}
display(Markdown(f"**Global training brightness temperature:** mean {calendar_stats['global_mean_k']:.1f} K; "
                 f"standard deviation {calendar_stats['global_std_k']:.1f} K."))
for label, manifest, stem in (
    ("Calendar-balanced training · 131,072 clips", calendar_manifests["Training · 131,072 clips"], "calendar_training_131072"),
    ("Calendar-balanced validation · 10,240 clips", calendar_manifests["Extended validation · 10,240 clips"], "calendar_validation_10240"),
):
    coverage, origins = views.manifest_coverage_figure(manifest, calendar_protocol["zarr_root"], title=label)
    figure(coverage, stem + "_coverage.png")
    origins.to_csv(OUTPUT / (stem + "_origins.csv"), index=False)
    pd.to_datetime(manifest.start_time, utc=True).dt.normalize().value_counts().sort_index().rename("clips").to_csv(
        OUTPUT / (stem + "_dates.csv"))

In [ ]:
month_counts = pd.DataFrame({
    label: pd.to_datetime(manifest.start_time, utc=True).dt.month.value_counts().reindex(range(1, 13), fill_value=0)
    for label, manifest in calendar_manifests.items()
})
month_percentages = month_counts.div(month_counts.sum(axis=0), axis=1) * 100
fig, axis = plt.subplots(figsize=(12, 4), layout="constrained")
positions = np.arange(12)
width = .25
for offset, (label, colour) in enumerate(zip(month_percentages.columns, ("#0072B2", "#D55E00", "#009E73"))):
    axis.bar(positions + (offset - 1) * width, month_percentages[label], width=width, label=label, color=colour)
axis.set(xticks=positions, xticklabels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
         xlabel="Month in 2022", ylabel="Clips in each split (%)", ylim=(0, 21), title="Monthly sampling contributions")
axis.legend(frameon=False)
axis.grid(axis="y", alpha=.2)
figure(fig, "calendar_monthly_contributions.png")
month_counts.to_csv(OUTPUT / "calendar_clips_by_month.csv")